# Batch Processing
## Name: Pablo Camarillo

In [89]:
from pcamarillor.spark_utils import SparkUtils

# Create Spark session
su = SparkUtils("Batch Processing", "spark://spark-master:7077")
sc = su._spark.sparkContext
sc


<SparkContext master=spark://spark-master:7077 appName=Batch Processing>

In [90]:
# Create the schema
"""
Order_ID,Company,City,Customer_Age,Order_Value,Delivery_Time_Min,Distance_Km,Items_Count,Product_Category,Payment_Method,Customer_Rating,Discount_Applied,Delivery_Partner_Rating
"""
columns = [("Order_ID", "int"),
           ("Company", "string"),
           ("City", "string"),
           ("Customer_Age", "int"),
           ("Order_Value", "float"),
           ("Delivery_Time_Min", "float"),
           ("Distance_Km", "float"),
           ("Items_Count", "float"),
           ("Product_Category", "string"),
           ("Payment_Method", "string"),
           ("Customer_Rating", "float"),
           ("Discount_Applied", "float"),
           ("Delivery_Partner_Rating", "float")           
           ]

qcommerce_schema = SparkUtils.generate_schema(columns)
qcommerce_df = su._spark.read \
                .option("header", "true") \
                .schema(qcommerce_schema) \
                .csv("/opt/spark/work-dir/data/qcommerce/qcommerce.csv")
qcommerce_df.printSchema()

root
 |-- Order_ID: integer (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer_Age: integer (nullable = true)
 |-- Order_Value: float (nullable = true)
 |-- Delivery_Time_Min: float (nullable = true)
 |-- Distance_Km: float (nullable = true)
 |-- Items_Count: float (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Customer_Rating: float (nullable = true)
 |-- Discount_Applied: float (nullable = true)
 |-- Delivery_Partner_Rating: float (nullable = true)



In [ ]:
n_records = qcommerce_df.count()
n_records

In [ ]:
from pyspark.sql.functions import count, when, isnull

qcommerce_df.select([count(when(isnull(c), c)).alias(c) for c in qcommerce_df.columns]).show()


In [ ]:
qcommerce_df.count()

In [ ]:
clean = qcommerce_df.dropna()
clean.select([count(when(isnull(c), c)).alias(c) for c in clean.columns]).show()

In [ ]:
clean.count()

In [ ]:
clean_fillna = qcommerce_df.fillna({
    'City': "uknown",
    'Items_Count': 0,
    'Payment_Method': "uknown",
    'Customer_Rating': 1.0,
    'Discount_Applied': 0.0,
    'Delivery_Partner_Rating': 0.0
})
clean_fillna.count()

In [ ]:
clean_fillna.select([count(when(isnull(c), c)).alias(c) for c in clean_fillna.columns]).show()

In [ ]:
df_1 = clean_fillna.withColumn("Paid_Tax", col("Order_Value") * 0.16)
df_1.show(5)

In [ ]:
df_2 = df_1.withColumn("Discount_Level", when(col("Discount_Applied") == 1.0, "Great Deal").otherwise("No Deal")).orderBy("Discount_Applied", ascending=False)
df_2.show()

In [ ]:
df_3 = df_2.withColumn("Delivery_SLA", when(col("Delivery_Time_Min") <= 15, "FAST") \
                                        .when((col("Delivery_Time_Min") > 15) & (col("Delivery_Time_Min") <= 20), "ON_TIME") \
                                        .when(col("Delivery_Time_Min") > 20, "LATE")) \
                                        .filter(col("Delivery_SLA") == "LATE") \
                                        .orderBy("Delivery_Time_Min", ascending=False) \
                                        .select("Order_ID", "Company", "City", "Delivery_Time_Min", "Delivery_SLA")
df_3.show()
                    

In [ ]:
from pyspark.sql.functions import avg, count
df_4 = df_2.withColumn("Effective_Order_Value", col("Order_Value") * (1 - col("Discount_Applied"))) \
            .withColumn("Price_Bucket", when(col("Effective_Order_Value") < 200, "LOW") \
                                        .when((col("Effective_Order_Value") >= 200) & (col("Effective_Order_Value") <= 500), "MEDIUM") \
                                        .when(col("Effective_Order_Value") > 500, "HIGH")) \
            .groupBy("Price_Bucket").agg(
                count("*").alias("Count_Price_Bucket"),
                avg("Effective_Order_Value").alias("Avg_Effective_Value")
            ).orderBy("Avg_Effective_Value", ascending=False)
df_4.show()

In [ ]:
qcommerce_df_taks3 = df_1.withColumn("Age_Group", when(col("Customer_Age") <= 25, "YOUNG") \
                                                .when((col("Customer_Age") > 25) & (col("Customer_Age") < 44), "ADULT") \
                                                .when(col("Customer_Age") >= 45, "SENIOR")) \
                                                .filter((col("Customer_Age") > 0) & (col("Customer_Age") < 100)) \
                                                .groupBy("Age_Group", "Product_Category").agg(
                                                    count("*").alias("count_age_group"),
                                                    avg("Order_Value").alias("avg_order_value")
                                                ) \
                                                .orderBy("Age_Group", ascending=False)
qcommerce_df_taks3.show()

In [91]:
sc.stop()